This notebook is designed to run on a **Google Colab T4 GPU**. It uses `llama-cpp-python` to load your GGUF model and run a comprehensive battery of 20 tests.

### **The "Expert Audit" Strategy**
The prompts are divided into four levels:
1.  **Level 1: Foundational Theory** (Testing NIST definitions).
2.  **Level 2: Threat Hunting & MITRE** (Testing tactical mapping).
3.  **Level 3: Technical Exploitation & Secure Coding** (Testing deep C++/Python logic).
4.  **Level 4: Principal Architect & Compliance** (Testing high-level strategy).


### **Step 1: Installation & Setup**
Run this cell to install the GGUF runner with CUDA support.

In [ ]:
!mkdir -p ../../models/final_gguf/cyber_expert_llama/
print('Please import the gguf model under models/final_gguf/cyber_expert_llama/')

In [ ]:
# Install llama-cpp-python with CUDA support for T4
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python
!pip install huggingface_hub

import os
from google.colab import drive
from llama_cpp import Llama

# Mount Drive to access your GGUF
#if not os.path.exists('/content/drive'):
#    drive.mount('/content/drive')


# --- SET YOUR MODEL PATH ---
# Adjust this to where your GGUF actually is
MODEL_PATH = "../../models/final_gguf/cyber_expert_llama/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf"

if not os.path.exists(MODEL_PATH):
    print(f"❌ ERROR: Model not found at {MODEL_PATH}")
else:
    print("✅ Model found. Loading into T4 GPU...")
    # Initialize the model
    llm = Llama(
        model_path=MODEL_PATH,
        n_gpu_layers=-1, # Load all layers to GPU
        n_ctx=8192,      # Context window
        n_batch=512,
        verbose=False
    )
    print("🚀 CyberExpert-8B Loaded Successfully!")

### **Step 2: The Inference Engine**
This function ensures the **Llama 3.1 chat template** is applied correctly to your fine-tuned model.

In [ ]:
def ask_expert(question):
    system_prompt = "You are a Principal Cybersecurity Architect. You provide technical analysis grounded in NIST SSDF and SP 800-53."

    # Llama 3.1 Instruct Template
    full_prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n### "

    output = llm(
        full_prompt,
        max_tokens=1024,
        stop=["<|eot_id|>"],
        temperature=0.1, # Keep it low for expert accuracy
        echo=False
    )

    response = output["choices"][0]["text"]
    return "### " + response # Prepend the header as we discussed for the template hack

### **Step 3: The Ultimate 20-Prompt Audit**
Run this cell to start the automated testing.

In [ ]:
test_prompts = [
    # --- LEVEL 1: EASY (Foundational) ---
    "What are the five functions of the NIST Cybersecurity Framework (CSF)?",
    "Explain the technical difference between a vulnerability and an exploit.",
    "Define NIST AC-2 and why it is critical for identity management.",
    "What is the difference between Symmetric and Asymmetric encryption in NIST FIPS standards?",
    "Briefly explain the 'Zero Trust' principle of 'Never Trust, Always Verify' within NIST SP 800-207.",

    # --- LEVEL 2: INTERMEDIATE (Threat Hunting) ---
    "Map a 'Process Injection' attack to the relevant MITRE ATT&CK Technique ID and suggest two detection methods.",
    "How does NIST SP 800-61 Rev 2 describe the 'Containment' phase of an incident?",
    "Write a technical description of a 'Living off the Land' (LotL) binary and how to monitor them on Windows.",
    "Analyze the security risk of an exposed .git directory on a public web server.",
    "Explain how an attacker might use 'Lateral Movement' after obtaining initial access via a phishing token.",

    # --- LEVEL 3: HARD (Deep Technical/Coding) ---
    "Review this code: 'void func(char *str) { char buf[16]; strcpy(buf, str); }'. Identify the NIST SSDF task that addresses this.",
    "Explain the technical execution flow of a 'Stack Pivot' during an exploit. How does ASLR mitigate this?",
    "Provide a Python snippet for a secure, NIST-compliant way to hash passwords using Argon2.",
    "Analyze the difference between a Heap Spray and a Stack-based Buffer Overflow at the memory register level.",
    "What is a 'Format String' vulnerability? Provide a vulnerable C snippet and the hardened version.",

    # --- LEVEL 4: SENIOR (Architecture & Compliance) ---
    "Explain how to implement NIST SC-7 (Boundary Protection) in a multi-tenant Kubernetes cluster using NetworkPolicies.",
    "Develop a high-level technical policy for 'Software Supply Chain Security' based on NIST SP 800-161.",
    "Analyze the security implications of using 'AssumeRole' in AWS for cross-account access. Which NIST control governs this?",
    "Explain the technical challenges of maintaining memory safety in legacy C++ systems without a full rewrite.",
    "Provide a technical assessment of 'Man-in-the-Middle' (MitM) risks in a Zero Trust Architecture that utilizes mTLS."
]

import time
from IPython.display import clear_output, Markdown

results = []

print("🏁 Starting Comprehensive Model Audit...")

for i, prompt in enumerate(test_prompts):
    print(f"[{i+1}/20] Processing: {prompt[:50]}...")
    start_time = time.time()
    response = ask_expert(prompt)
    end_time = time.time()

    results.append({
        "level": (i // 5) + 1,
        "prompt": prompt,
        "response": response,
        "time": round(end_time - start_time, 2)
    })

clear_output()
print("✅ Audit Complete! Results rendered below:")

# Display formatted results
for res in results:
    display(Markdown(f"---"))
    display(Markdown(f"**Level {res['level']} Test** (Response time: {res['time']}s)"))
    display(Markdown(f"**User:** {res['prompt']}"))
    display(Markdown(f"**Expert Assistant:**\n{res['response']}"))

### **Step 4: Final Evaluation & Performance Review**
Run this cell to see your final stats and save the results for your documentation.


In [ ]:
import json

# Save to JSON for your records
with open("expert_audit_results.json", "w") as f:
    json.dump(results, f, indent=4)

avg_time = sum(r['time'] for r in results) / 20
print(f"📊 Average Response Time: {avg_time} seconds")
print(f"📝 Full report saved to: expert_audit_results.json")

# Guidance on what to check
print("\n🔍 HOW TO GRADE YOUR EXPERT:")
print("1. NIST Mapping: Did it correctly identify AC-2, SI-16, etc.?")
print("2. Formatting: Is the output properly structured with headers?")
print("3. Technical Depth: Did it provide CLI commands or memory register details?")
print("4. Hallucination: Check if the NIST control IDs exist!")